# Depth Anything V2 — Depth-Only Backend API

This notebook runs **only Depth Anything V2 Metric (ViT-S)** in Google Colab.

- **Object detection (YOLO11)** runs **on-device** inside the Flutter app (`yolo11n.tflite`).
- The Flutter app sends the camera frame + its detected bounding boxes (`left, top, right, bottom` normalized `0..1`) to `/estimate-depth`.
- This backend computes calibrated distances (`calibrated_distance_m`) for each box and returns them to the app.

In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd /content/Depth-Anything-V2
!pip install -q -r metric_depth/requirements.txt
print('Depth Anything V2 dependencies installed (no YOLO / ultralytics).')

In [ ]:
import os, urllib.request

CHECKPOINT_DIR = "/content/Depth-Anything-V2/metric_depth/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "depth_anything_v2_metric_hypersim_vits.pth")
URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Hypersim-Small/resolve/main/depth_anything_v2_metric_hypersim_vits.pth"

if not os.path.exists(CHECKPOINT_PATH):
    urllib.request.urlretrieve(URL, CHECKPOINT_PATH)

print("Size MB:", round(os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024), 2))

In [ ]:
import sys, importlib, torch

metric_path = "/content/Depth-Anything-V2/metric_depth"
if metric_path not in sys.path:
    sys.path.insert(0, metric_path)
importlib.invalidate_caches()

from depth_anything_v2.dpt import DepthAnythingV2

depth_model = DepthAnythingV2(
    encoder="vits", features=64,
    out_channels=[48, 96, 192, 384], max_depth=20
)
depth_model.load_state_dict(
    torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True),
    strict=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
depth_model = depth_model.to(device).eval()
print("Model loaded on", device)

In [ ]:
import numpy as np

INPUT_SIZE = 256
INSET_RATIO = 0.20

def generate_depth_map(image_bgr):
    if image_bgr is None:
        raise ValueError("Image is empty.")

    depth = depth_model.infer_image(image_bgr, INPUT_SIZE)
    return np.asarray(depth, dtype=np.float32)

print("Depth model is ready")

In [ ]:
def calculate_median_depth(depth_map, bbox, inset_ratio=INSET_RATIO):

    height, width = depth_map.shape[:2]

    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, min(x1, width))
    x2 = max(0, min(x2, width))
    y1 = max(0, min(y1, height))
    y2 = max(0, min(y2, height))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid bounding box.")

    inset_x = int((x2 - x1) * inset_ratio)
    inset_y = int((y2 - y1) * inset_ratio)

    region = depth_map[
        y1 + inset_y:y2 - inset_y,
        x1 + inset_x:x2 - inset_x
    ]

    if region.size == 0:
        region = depth_map[y1:y2, x1:x2]

    valid = region[np.isfinite(region) & (region > 0)]

    if valid.size == 0:
        raise ValueError("No valid depth values found.")

    return float(np.median(valid))

print("Median depth function ready")

In [ ]:
SCALE_FACTOR = 0.444  # Mathematically calibrated to map 4.5m raw depth down to 2.0m real depth
OFFSET = 0.0

def calibrate_distance(raw_depth, scale=SCALE_FACTOR, offset=OFFSET):
    # The Metric Depth Anything model outputs direct distance, but requires global scaling.
    calibrated_distance = (raw_depth * scale) + offset
    return float(max(0.0, calibrated_distance))

print("Calibration loaded successfully!")
print("Scale Factor:", SCALE_FACTOR)
print("Offset:", OFFSET)

In [ ]:
def fit_calibration(raw_depths, actual_distances):
    """
    Calculate calibration slope and intercept
    from raw depth values and actual measured distances.
    """
    import numpy as np
    raw_depths = np.asarray(raw_depths, dtype=np.float64)
    actual_distances = np.asarray(actual_distances, dtype=np.float64)

    if len(raw_depths) != len(actual_distances):
        raise ValueError("Both lists must have the same length.")

    if len(raw_depths) < 2:
        raise ValueError("At least two calibration samples are required.")

    slope, intercept = np.polyfit(
        raw_depths, # Direct linear fit
        actual_distances,
        1
    )

    print("Slope:", slope)
    print("Intercept:", intercept)

    return float(slope), float(intercept)

print("Calibration fitting function is ready!")

In [ ]:
def estimate_distance(depth_map, bbox, scale, offset):
    raw_depth = calculate_median_depth(depth_map, bbox)

    calibrated = calibrate_distance(
        raw_depth,
        scale,
        offset
    )

    return {
        "raw_depth_m": raw_depth,
        "calibrated_distance_m": calibrated
    }


def process_detection(image_bgr, bbox, scale, offset):
    depth_map = generate_depth_map(image_bgr)

    return estimate_distance(
        depth_map,
        bbox,
        scale,
        offset
    )


print("Distance estimation pipeline is ready!")

In [ ]:
import numpy as np

def normalize_bbox_to_pixels(det, img_w, img_h):
    """
    Accepts either:
      - Flutter Detection format (normalized 0..1): {"left": ..., "top": ..., "right": ..., "bottom": ...}
      - Pixel bbox list/tuple: {"bbox": [x1, y1, x2, y2]}
    Returns (x1, y1, x2, y2) clamped to image bounds.
    """
    if all(k in det for k in ("left", "top", "right", "bottom")):
        l, t, r, b = float(det["left"]), float(det["top"]), float(det["right"]), float(det["bottom"])
        # Auto-detect normalized (0..1) vs pixel coordinates
        if max(abs(l), abs(t), abs(r), abs(b)) <= 1.5:
            x1, y1 = int(round(l * img_w)), int(round(t * img_h))
            x2, y2 = int(round(r * img_w)), int(round(b * img_h))
        else:
            x1, y1, x2, y2 = int(round(l)), int(round(t)), int(round(r)), int(round(b))
    elif "bbox" in det and len(det["bbox"]) == 4:
        x1, y1, x2, y2 = [int(round(v)) for v in det["bbox"]]
    else:
        raise ValueError(f"Unsupported bounding box format: {det}")

    x1 = max(0, min(img_w - 1, x1))
    y1 = max(0, min(img_h - 1, y1))
    x2 = max(x1 + 1, min(img_w, x2))
    y2 = max(y1 + 1, min(img_h, y2))
    return (x1, y1, x2, y2)


def estimate_distances_for_boxes(image_bgr, incoming_detections):
    """
    Runs ONLY Depth Anything V2 on image_bgr and attaches distance estimates
    to the bounding boxes already detected on-device by the Flutter app.
    """
    if image_bgr is None:
        raise ValueError("Input image is empty.")

    if not incoming_detections:
        return [], None

    img_h, img_w = image_bgr.shape[:2]
    depth_map = generate_depth_map(image_bgr)

    results = []
    for det in incoming_detections:
        bbox_px = normalize_bbox_to_pixels(det, img_w, img_h)
        try:
            dist_info = estimate_distance(depth_map, bbox_px, SCALE_FACTOR, OFFSET)
            raw_depth_m = dist_info["raw_depth_m"]
            calibrated_m = dist_info["calibrated_distance_m"]
        except ValueError:
            raw_depth_m = None
            calibrated_m = None

        # Keep all fields sent by the Flutter app intact and add distance fields
        item = dict(det)
        item["bbox_px"] = list(bbox_px)
        item["raw_depth_m"] = round(float(raw_depth_m), 3) if raw_depth_m is not None else None
        item["calibrated_distance_m"] = round(float(calibrated_m), 2) if calibrated_m is not None else None
        results.append(item)

    return results, depth_map

print("Depth-only bounding box distance estimator is ready!")


In [ ]:
import cv2
import numpy as np

def check_camera_visibility(frame, dark_thresh=35.0, blur_laplacian_thresh=15.0, edge_std_thresh=12.0):
    """
    Check frame brightness and texture variance to detect if the camera is
    in the dark or physically blocked/covered.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(np.mean(gray))
    std_brightness = float(np.std(gray))
    laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    is_dark = mean_brightness < dark_thresh
    is_blocked = (std_brightness < edge_std_thresh) and (laplacian_var < blur_laplacian_thresh)

    status = "ok"
    warning_message = None
    if is_dark:
        status = "dark"
        warning_message = "Camera environment is too dark."
    elif is_blocked:
        status = "blocked"
        warning_message = "Camera lens appears blocked or covered."

    return {
        "status": status,
        "is_dark": is_dark,
        "is_blocked": is_blocked,
        "mean_brightness": round(mean_brightness, 2),
        "warning_message": warning_message,
    }


## Real-time Depth-Only API (For On-Device Flutter YOLO Integration)

In [ ]:
!pip install -q fastapi uvicorn python-multipart nest-asyncio

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import JSONResponse
from fastapi.concurrency import run_in_threadpool
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image, ImageOps
import io
import json
import time
import threading
import cv2
import numpy as np

MAX_FRAME_DIM = 640
processing_lock = threading.Lock()
total_frames_processed = 0
total_processing_time = 0.0
_BILINEAR = getattr(Image, "Resampling", Image).BILINEAR

def decode_phone_image(image_bytes, max_dim=MAX_FRAME_DIM):
    try:
        pil_img = Image.open(io.BytesIO(image_bytes))
        pil_img = ImageOps.exif_transpose(pil_img).convert("RGB")
        w, h = pil_img.size
        longest = max(w, h)
        if longest > max_dim:
            scale = max_dim / float(longest)
            pil_img = pil_img.resize((int(w * scale), int(h * scale)), _BILINEAR)
        rgb = np.array(pil_img)
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    except Exception as e:
        raise ValueError(f"Failed to decode image: {e}")

app = FastAPI(title="Depth Anything V2 — Depth-Only API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    avg_ms = (total_processing_time / total_frames_processed * 1000.0) if total_frames_processed > 0 else 0.0
    return {
        "status": "ok",
        "depth_model": "Depth-Anything-V2-Metric-Hypersim-Small",
        "yolo_in_colab": False,
        "frames_processed": total_frames_processed,
        "avg_latency_ms": round(avg_ms, 1),
    }

@app.post("/estimate-depth")
async def estimate_depth_endpoint(
    file: UploadFile = File(...),
    detections: str = Form("[]")
):
    """
    Accepts:
      - file: camera image (JPEG/PNG)
      - detections: JSON array of bounding boxes from the Flutter app's on-device YOLO, e.g.:
        [{"classId": 0, "label": "person", "score": 0.87, "left": 0.2, "top": 0.1, "right": 0.6, "bottom": 0.9, "isObstacle": true}]
    Returns the same detections enriched with raw_depth_m and calibrated_distance_m.
    """
    global total_frames_processed, total_processing_time

    image_bytes = await file.read()
    if not image_bytes:
        raise HTTPException(status_code=400, detail="Empty image upload")

    try:
        incoming_detections = json.loads(detections) if detections else []
        if not isinstance(incoming_detections, list):
            raise ValueError("detections must be a JSON list")
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid detections JSON: {exc}")

    def _process():
        start = time.perf_counter()
        frame = decode_phone_image(image_bytes)
        visibility = check_camera_visibility(frame)
        if visibility["is_blocked"] or visibility["is_dark"]:
            elapsed = time.perf_counter() - start
            return [], visibility, elapsed
        with processing_lock:
            enriched, _ = estimate_distances_for_boxes(frame, incoming_detections)
        elapsed = time.perf_counter() - start
        return enriched, visibility, elapsed

    try:
        enriched, visibility, elapsed = await run_in_threadpool(_process)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))

    total_frames_processed += 1
    total_processing_time += elapsed

    return JSONResponse({
        "camera_visibility": visibility,
        "latency_ms": round(elapsed * 1000.0, 1),
        "detections": enriched,
    })

print("Depth-Only FastAPI app is ready! Endpoint: POST /estimate-depth")


In [ ]:
import os, urllib.request, threading, time, uvicorn, nest_asyncio, re

nest_asyncio.apply()

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

if "api_server_thread" not in globals() or not api_server_thread.is_alive():
    def _run_api():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
    api_server_thread = threading.Thread(target=_run_api, daemon=True)
    api_server_thread.start()
    time.sleep(3)

try:
    with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as response:
        print("Local API:", response.read().decode())
except Exception as exc:
    raise RuntimeError("FastAPI did not start.") from exc

# Safely start Cloudflare in the background and read from a log file
!nohup /usr/local/bin/cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &

print("Waiting 6 seconds for Cloudflare to generate the URL...")
time.sleep(6)

if os.path.exists("cloudflared.log"):
    with open("cloudflared.log", "r") as f:
        log_content = f.read()

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)

    if match:
        public_url = match.group(0)
        print("\n✅ API PUBLIC URL:", public_url)
        print("Health check:", public_url + "/health")
        print("Flutter depth endpoint:", public_url + "/estimate-depth")
    else:
        print("⚠️ Could not find Cloudflare URL. Here is the exact log output to see what went wrong:")
        print("--- START LOG ---")
        print(log_content)
        print("--- END LOG ---")
else:
    print("Error: Cloudflare log file was not created.")

## Optional: Depth Anything V2 ONNX Export (Depth Model Only)

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import torch
import onnx
import onnxruntime as ort
import numpy as np

# Prepare the loaded model
depth_model = depth_model.cpu().eval()

class ExportModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, image):
        return self.model(image)

export_model = ExportModel(depth_model).eval()

# Input dimensions: 336x252 (3:4 portrait, exact multiple of 14, optimized for Exynos 1380)
ONNX_H, ONNX_W = 336, 252
dummy_input = torch.randn(1, 3, ONNX_H, ONNX_W)

ONNX_PATH = "/content/depth_anything_metric.onnx"

with torch.no_grad():
    torch.onnx.export(
        export_model,
        dummy_input,
        ONNX_PATH,
        input_names=["image"],
        output_names=["depth"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False
    )

# Validate ONNX model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

# Test inference
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

test_input = np.random.randn(
    1, 3, ONNX_H, ONNX_W
).astype(np.float32)

result = session.run(
    ["depth"],
    {"image": test_input}
)

print("ONNX export successful!")
print("Output shape:", result[0].shape)
print("Model size MB:", round(
    __import__("os").path.getsize(ONNX_PATH) / (1024 * 1024), 2
))

In [ ]:
# Compare PyTorch and ONNX outputs using the same input

test_input = np.random.randn(
    1, 3, ONNX_H, ONNX_W
).astype(np.float32)

with torch.no_grad():
    torch_result = depth_model(
        torch.from_numpy(test_input)
    ).cpu().numpy()

onnx_result = session.run(
    ["depth"],
    {"image": test_input}
)[0]

max_diff = np.max(
    np.abs(torch_result - onnx_result)
)

mean_diff = np.mean(
    np.abs(torch_result - onnx_result)
)

print("PyTorch shape:", torch_result.shape)
print("ONNX shape:", onnx_result.shape)
print("Maximum difference:", max_diff)
print("Mean difference:", mean_diff)

if np.allclose(
    torch_result,
    onnx_result,
    atol=1e-3,
    rtol=1e-3
):
    print("SUCCESS: ONNX matches PyTorch!")
else:
    print("WARNING: Outputs are different!")

# Restore the model to the selected runtime device for live frame processing.
depth_model = depth_model.to(device).eval()
print("Depth model restored to", device)